<a href="https://colab.research.google.com/github/Parthmagdum/Bobble-AI/blob/main/Assignment_No_5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import re
from nltk.stem import PorterStemmer

# 1. Load the dataset
file_path = "/content/drive/MyDrive/Untitled folder/product_reviews.csv"  # Make sure file is in your working directory
df = pd.read_csv(file_path, encoding="latin1")

In [3]:
# 2. Convert all text to lowercase
df['Review_Text'] = df['Review_Text'].str.lower()

In [4]:
# 3. Remove punctuation, numbers, and special characters
df['Review_Text'] = df['Review_Text'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))

In [5]:
# 4. Tokenize into individual words
df['tokens'] = df['Review_Text'].apply(lambda x: re.split(r'\s+', x.strip()))

In [6]:
# 5. Remove stopwords (manual list, works offline)
stopwords_list = set("""
a about above after again against all am an and any are aren't as at be because been before
being below between both but by can't cannot could couldn't did didn't do does doesn't doing
don't down during each few for from further had hadn't has hasn't have haven't having he he'd
he'll he's her here here's hers herself him himself his how how's i i'd i'll i'm i've if in
into is isn't it it's its itself let's me more most mustn't my myself no nor not of off on once
only or other ought our ours ourselves out over own same shan't she she'd she'll she's should
shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was
wasn't we we'd we'll we're we've were weren't what what's when when's where where's which while
who who's whom why why's with won't would wouldn't you you'd you'll you're you've your yours
yourself yourselves
""".split())

df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stopwords_list and word != ''])

In [7]:
# 6. Perform stemming
stemmer = PorterStemmer()
df['stemmed'] = df['tokens'].apply(lambda x: [stemmer.stem(word) for word in x])

In [8]:
# 7. Lemmatization (fallback to same tokens for offline use)
df['lemmatized'] = df['tokens']

In [9]:
# 8. Custom TF-IDF calculation
def custom_tf(word, doc_tokens):
    count = doc_tokens.count(word)  # C(w,d)
    return 1 + np.log(count + 1)    # 1 + log(C(w,d) + 1)

In [10]:
# Build vocabulary
vocab = sorted(set(word for tokens in df['lemmatized'] for word in tokens))
N = len(df)  # Total number of documents

In [11]:
# Calculate TF-IDF for each review
tfidf_results = []
for tokens in df['lemmatized']:
    row_scores = {}
    for word in vocab:
        tf = custom_tf(word, tokens)
        df_count = sum(1 for doc in df['lemmatized'] if word in doc)
        idf = np.log(N / (1 + df_count))
        row_scores[word] = tf * idf
    tfidf_results.append(row_scores)

In [12]:
# Convert TF-IDF results to DataFrame
tfidf_df = pd.DataFrame(tfidf_results)

In [13]:
# Show processed data
print("\nProcessed Tokens, Stems, and Lemmas:")
print(df[['Review_Text', 'tokens', 'stemmed', 'lemmatized']].head())

print("\nTF-IDF Matrix:")
print(tfidf_df.head())


Processed Tokens, Stems, and Lemmas:
                                         Review_Text  \
0  the product is great loved it but its a bit pr...   
1     worst product ever wouldnt recommend to anyone   
2  satisfactory quality works as expected no majo...   
3     amazing product i would buy it again and again   
4      the delivery was slow but the product is good   

                                              tokens  \
0               [product, great, loved, bit, pricey]   
1  [worst, product, ever, wouldnt, recommend, any...   
2  [satisfactory, quality, works, expected, major...   
3                            [amazing, product, buy]   
4                    [delivery, slow, product, good]   

                                             stemmed  \
0                [product, great, love, bit, pricey]   
1  [worst, product, ever, wouldnt, recommend, anyon]   
2  [satisfactori, qualiti, work, expect, major, i...   
3                               [amaz, product, buy]   
4       